# MESC Experiment-0 — Colab Foundation Tournament

**Protocol status:** preparation template only.

This notebook is a fail-closed execution surface for the future no-training MESC foundation tournament.
It is not runtime evidence until run under a canonical frozen `MESC-EXPERIMENT-0-CONFIG-V1`
whose MRL-0801..MRL-0899 authority/evidence bindings are genuine and current.

Do not edit cells to bypass a blocked gate. Do not paste credentials into source or output.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

CONFIG_PATH = Path("/content/experiment-config.json")
ROOT = Path("/content/mesc-experiment-0")
REPO = ROOT / "repo"
EVIDENCE = ROOT / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)

CONFIG_SCHEMA = "MESC-EXPERIMENT-0-CONFIG-V1"
RUNTIME_SCHEMA = "MESC-EXPERIMENT-0-RUNTIME-V1"
ENVIRONMENT_SCHEMA = "MESC-EXPERIMENT-0-ENVIRONMENT-V1"
REQUIRED_AUTHORITY_KEYS = (
    "mrl_0801_evidence_id",
    "mrl_0802_evidence_id",
    "mrl_0803_evidence_id",
    "mrl_0804_evidence_id",
    "mrl_0805_authority_id",
    "mrl_0806_objective_id",
    "mrl_0807_evaluator_freeze_id",
    "mrl_0808_sandbox_id",
    "mrl_0809_preflight_id",
    "mrl_0899_readiness_id",
)
REQUIRED_FROZEN_FIELDS = (
    "experiment_id",
    "objective_id",
    "repository_sha",
    "repository_tree",
    "strategy_decision_id",
    "candidate_roster",
    "dataset_identities",
    "evaluator_identities",
    "prompt_template_identities",
    "generation_configs",
    "runtime_policy",
    "network_policy",
    "filesystem_policy",
    "credential_policy",
    "resource_budget",
    "query_budget",
    "result_exposure_budget",
    "hard_floor_policy",
    "decision_rule",
    "sealed_evaluation_policy",
    "authority_bindings",
)


def canonical_json_bytes(value: object) -> bytes:
    """Serialize deterministic UTF-8 JSON bytes for hashing and evidence files."""
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def sha256_bytes(data: bytes) -> str:
    """Return the lowercase SHA-256 digest of exact bytes."""
    return hashlib.sha256(data).hexdigest()


def write_json(path: Path, value: object) -> str:
    """Write exact canonical JSON bytes and return the digest of those stored bytes."""
    data = canonical_json_bytes(value)
    path.write_bytes(data)
    return sha256_bytes(data)


def blocked(reason: str, **facts: object) -> None:
    """Persist a metadata-only blocked receipt and stop execution."""
    payload = {
        "schema_version": "MESC-EXPERIMENT-0-BLOCKED-V1",
        "reason": reason,
        "facts": facts,
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(EVIDENCE / "blocked.json", payload)
    raise RuntimeError(f"MESC Experiment-0 BLOCKED: {reason}")


In [ ]:
# Load and validate the exact frozen config before any network/model access.
if not CONFIG_PATH.exists():
    blocked("FROZEN_CONFIG_MISSING", expected_path=str(CONFIG_PATH))

try:
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
except (UnicodeDecodeError, json.JSONDecodeError) as exc:
    blocked("FROZEN_CONFIG_INVALID_JSON", failure_class=type(exc).__name__)

if not isinstance(config, dict):
    blocked("FROZEN_CONFIG_NOT_OBJECT")
if config.get("schema_version") != CONFIG_SCHEMA:
    blocked("CONFIG_SCHEMA_MISMATCH", observed=config.get("schema_version"))
if config.get("status") != "FROZEN_EXECUTION_CONFIG":
    blocked("CONFIG_NOT_FROZEN", observed=config.get("status"))

missing_fields = [field for field in REQUIRED_FROZEN_FIELDS if field not in config]
if missing_fields:
    blocked("CONFIG_REQUIRED_FIELD_MISSING", fields=missing_fields)

for field in ("experiment_id", "objective_id", "repository_sha", "repository_tree", "strategy_decision_id"):
    if not isinstance(config.get(field), str) or not config[field].strip():
        blocked("CONFIG_REQUIRED_FIELD_INVALID", field=field)

for field in ("candidate_roster", "dataset_identities", "evaluator_identities", "prompt_template_identities", "generation_configs"):
    value = config.get(field)
    if not isinstance(value, list) or not value:
        blocked("CONFIG_FROZEN_LIST_INVALID", field=field)

for field in ("runtime_policy", "network_policy", "filesystem_policy", "credential_policy", "resource_budget", "query_budget", "result_exposure_budget", "hard_floor_policy", "decision_rule", "sealed_evaluation_policy", "authority_bindings"):
    value = config.get(field)
    if not isinstance(value, dict) or not value:
        blocked("CONFIG_FROZEN_OBJECT_INVALID", field=field)

bindings = config["authority_bindings"]
missing_bindings = [key for key in REQUIRED_AUTHORITY_KEYS if not bindings.get(key)]
if missing_bindings:
    blocked("MRL_AUTHORITY_OR_EVIDENCE_BINDING_MISSING", missing=missing_bindings)

for section, keys in {
    "resource_budget": ("max_gpu_hours", "max_wall_hours", "max_storage_bytes", "max_retries"),
    "query_budget": ("max_adaptive_queries",),
    "result_exposure_budget": ("tier1_max_exposures", "tier2_max_exposures", "tier3_allowed_fields"),
}.items():
    missing_budget_values = [key for key in keys if config[section].get(key) is None]
    if missing_budget_values:
        blocked("FROZEN_BUDGET_VALUE_MISSING", section=section, fields=missing_budget_values)

sealed_policy = config["sealed_evaluation_policy"]
if sealed_policy.get("tier3_item_access_by_research_process") is not False:
    blocked("SEALED_TIER3_POLICY_INVALID")

runtime_policy = config.get("runtime_policy")
if not isinstance(runtime_policy, dict):
    blocked("RUNTIME_POLICY_MISSING_OR_INVALID")
allowed_gpu_count = runtime_policy.get("allowed_gpu_count")
if isinstance(allowed_gpu_count, bool) or not isinstance(allowed_gpu_count, int) or allowed_gpu_count < 1:
    blocked("RUNTIME_POLICY_GPU_COUNT_INVALID", observed=allowed_gpu_count)
if not isinstance(runtime_policy.get("allowed_gpu_models"), list):
    blocked("RUNTIME_POLICY_GPU_MODELS_INVALID")
if not isinstance(runtime_policy.get("allow_unlisted_gpu_model"), bool):
    blocked("RUNTIME_POLICY_ALLOW_UNLISTED_INVALID")

config_sha256 = sha256_bytes(canonical_json_bytes(config))
print(json.dumps({
    "config_schema": CONFIG_SCHEMA,
    "experiment_id": config["experiment_id"],
    "config_sha256": config_sha256,
    "candidate_count": len(config["candidate_roster"]),
}, indent=2))


In [ ]:
# Attest the actual Google-hosted GPU runtime. Do not assume GPU class.
runtime_started = datetime.now(timezone.utc).isoformat()

try:
    import google.colab  # type: ignore  # noqa: F401
    colab_present = True
except Exception:
    colab_present = False

if not colab_present:
    blocked("NOT_GOOGLE_COLAB_HOSTED_RUNTIME")

try:
    import torch
except Exception as exc:
    blocked(
        "TORCH_IMPORT_FAILED",
        failure_class=type(exc).__name__,
        failure_message_sha256=sha256_bytes(str(exc).encode("utf-8", errors="replace")),
    )

if not torch.cuda.is_available():
    blocked("CUDA_UNAVAILABLE")

gpu_count = int(torch.cuda.device_count())
expected_count = runtime_policy["allowed_gpu_count"]
if gpu_count != expected_count:
    blocked("GPU_COUNT_POLICY_MISMATCH", observed=gpu_count, expected=expected_count)

gpu_models = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
gpu_total_memory = [int(torch.cuda.get_device_properties(i).total_memory) for i in range(gpu_count)]
allowed_gpu_models = list(runtime_policy["allowed_gpu_models"])
allow_unlisted = runtime_policy["allow_unlisted_gpu_model"]
if allowed_gpu_models and not allow_unlisted:
    unexpected = [name for name in gpu_models if name not in allowed_gpu_models]
    if unexpected:
        blocked("GPU_MODEL_NOT_IN_FROZEN_POLICY", observed=gpu_models, allowed=allowed_gpu_models)

print(json.dumps({
    "runtime_provider": "GOOGLE_COLAB",
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_count": gpu_count,
    "gpu_models": gpu_models,
    "gpu_total_memory_bytes": gpu_total_memory,
}, indent=2))


In [ ]:
# Bind execution to the exact canonical repository commit/tree.
if REPO.exists():
    blocked("REPOSITORY_PATH_ALREADY_EXISTS", path=str(REPO))

network_policy = config["network_policy"]
if network_policy.get("allow_repository_clone") is not True:
    blocked("REPOSITORY_CLONE_NOT_ALLOWED_BY_FROZEN_POLICY")

clone = subprocess.run(
    ["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/TheHalfMoon/MESC.git", str(REPO)],
    text=True,
    capture_output=True,
)
if clone.returncode != 0:
    blocked(
        "REPOSITORY_CLONE_FAILED",
        returncode=clone.returncode,
        stderr_sha256=sha256_bytes(clone.stderr.encode("utf-8", errors="replace")),
    )

checkout = subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", config["repository_sha"]],
    text=True,
    capture_output=True,
)
if checkout.returncode != 0:
    blocked(
        "REPOSITORY_CHECKOUT_FAILED",
        returncode=checkout.returncode,
        stderr_sha256=sha256_bytes(checkout.stderr.encode("utf-8", errors="replace")),
    )

observed_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
observed_tree = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD^{tree}"], text=True).strip()
if observed_head != config["repository_sha"]:
    blocked("REPOSITORY_SHA_MISMATCH", observed=observed_head, expected=config["repository_sha"])
if observed_tree != config["repository_tree"]:
    blocked("REPOSITORY_TREE_MISMATCH", observed=observed_tree, expected=config["repository_tree"])

print(json.dumps({"repository_sha": observed_head, "repository_tree": observed_tree}, indent=2))


In [ ]:
# Record package name/version metadata only; do not persist direct URLs or credentials.
packages = []
for distribution in importlib.metadata.distributions():
    name = distribution.metadata.get("Name")
    version = distribution.version
    if name and version:
        packages.append({"name": str(name), "version": str(version)})
packages = sorted(packages, key=lambda item: (item["name"].casefold(), item["version"]))

environment_manifest = {
    "schema_version": ENVIRONMENT_SCHEMA,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": packages,
}
environment_sha256 = write_json(EVIDENCE / "environment-manifest.json", environment_manifest)

try:
    transformers_version = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    transformers_version = None

runtime_receipt = {
    "schema_version": RUNTIME_SCHEMA,
    "experiment_config_sha256": config_sha256,
    "repository_sha": observed_head,
    "repository_tree": observed_tree,
    "execution_started_at_utc": runtime_started,
    "execution_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_provider": "GOOGLE_COLAB",
    "runtime_class": "GOOGLE_COLAB_HOSTED_GPU_RUNTIME",
    "python_version": platform.python_version(),
    "platform_string": platform.platform(),
    "torch_version": torch.__version__,
    "transformers_version": transformers_version,
    "cuda_available": True,
    "cuda_version": torch.version.cuda,
    "gpu_count": gpu_count,
    "gpu_models": gpu_models,
    "gpu_total_memory_bytes": gpu_total_memory,
    "colab_release_tag_or_image_identity_if_observable": os.environ.get("COLAB_RELEASE_TAG"),
    "installed_environment_manifest_sha256": environment_sha256,
    "network_policy_observation": "FROZEN_PREPARATION_NETWORK_POLICY_OBSERVED",
    "credential_surface_observation": "NO_CREDENTIAL_READ_OR_PERSISTENCE_IN_PREPARATION_TEMPLATE",
    "final_runtime_disposition": "PASS_RUNTIME_PREFLIGHT",
    "stop_reason": None,
}
runtime_sha256 = write_json(EVIDENCE / "runtime-receipt.json", runtime_receipt)
print(json.dumps({
    "runtime_receipt_sha256": runtime_sha256,
    "environment_manifest_sha256": environment_sha256,
}, indent=2))


In [ ]:
# Intentional preparation stop.
# Candidate model acquisition and medical evaluation adapters are not guessed here.
blocked(
    "CANDIDATE_EXECUTION_ADAPTERS_NOT_CANONICAL_YET",
    runtime_receipt_sha256=runtime_sha256,
    note="Preparation template completed frozen-config/runtime/repository attestation only.",
)
